# AI Production Scheduler — two-week simulation

This notebook implements the decision-support scheduler using the supplied
historical employee scores. It does not recalculate performance. It enforces
business workflows, working hours, approved leaves, a 20-minute post-task
break, and a maximum of 38 hours per ISO week.

Two-stage projects split their fixed effort 45% Redaction / 55% Graphe so
their total workload remains the duration defined by the difficulty mapping.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import re
import unicodedata
import os
import requests
import urllib3

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)

# The notebook runs from your medimar folder, so simple filenames are enough.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

SIMULATION_DAYS = 14
MAX_WEEKLY_HOURS = 38.0
POST_TASK_BREAK_MINUTES = 20
WORKING_PERIODS = ((8, 30, 12, 0), (13, 30, 17, 0))
STAGE_EFFORT_SPLIT = {"Redaction": 0.45, "Graphe": 0.55}

TRIWEB_AVAILABILITY_API = "https://tools.triweb-apps.com/Triweb_NewV/intern/api/VPlanificationV/GetVwCollaboratorDailyAvailability"
TRIWEB_PROJECTS_API = "https://tools.triweb-apps.com/Triweb_NewV/intern/api/VPlanificationV"
# Set TRIWEB_VERIFY_TLS=true after your internal CA is installed in Python.
VERIFY_TLS = os.getenv("TRIWEB_VERIFY_TLS", "false").strip().lower() in {"1", "true", "yes"}
if not VERIFY_TLS:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


## 1. Business configuration

Every difficulty in the supplied project data has a fixed duration in hours.
The greater-than-three-hour correction categories are modelled as four hours;
change these standards here if the business provides revised effort estimates.

In [ ]:
def normal(value: object) -> str:
    """Return an accent-insensitive key for matching business labels."""
    text = "" if pd.isna(value) else str(value)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower().replace(">", " greater than ")
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


_duration_labels = {
    **{f"Création de page ({hour}H)": float(hour) for hour in range(1, 12)},
    "Corrections CREA/minimes (-30 min)": 0.5,
    "Corrections CREA/importantes (1h)": 1.0,
    "Corrections CREA/importantes (1h30 )": 1.5,
    "Corrections CREA/importantes (2H )": 2.0,
    "Corrections CREA/importantes (2H30 )": 2.5,
    "Corrections CREA/importantes (3H )": 3.0,
    "Corrections CREA/importantes (> 3H )": 4.0,
    "Corrections MAJ/minimes (-30 min)": 0.5,
    "Corrections MAJ/importantes (1h )": 1.0,
    "Corrections MAJ/importantes (1h30 )": 1.5,
    "Corrections MAJ/importantes (2H )": 2.0,
    "Corrections MAJ/importantes (2H30 )": 2.5,
    "Corrections MAJ/importantes (3H )": 3.0,
    "Corrections MAJ/importantes (> 3H )": 4.0,
    "Création du logo": 3.0,
    "Maquette": 6.0,
    "Création du site Webtool": 8.0,
    "Création du site Toolbox": 8.0,
    "Rédactionnel / création site ecommerce": 10.0,
    "Oxatis to Wizishop": 8.0,
    "Fiche GMB / Facebook / Visibilité": 2.0,
    "Référencement/SEO (30 min)": 0.5,
    "Référencement/SEO (1h)": 1.0,
    "Référencement/SEO (1h30)": 1.5,
    "Référencement/SEM": 1.0,
    "Référencement/SEA": 1.0,
}
DURATION_BY_DIFFICULTY = {normal(label): hours for label, hours in _duration_labels.items()}

WORKFLOW_BY_SKILL = {
    normal(skill): stages
    for skill, stages in {
        "Creation Page": ("Redaction", "Graphe"),
        "Webtool": ("Redaction", "Graphe"),
        "Toolbox": ("Redaction", "Graphe"),
        "Ecommerce": ("Redaction", "Graphe"),
        "GMB/Facebook": ("Redaction", "Graphe"),
        "Oxatis": ("Redaction", "Graphe"),
        "Corrections CREA": ("Redaction", "Graphe"),
        "Corrections MAJ": ("Redaction", "Graphe"),
        "SEO": ("Redaction",),
        "SEM/SEA": ("Redaction",),
        "Logo": ("Graphe",),
        "Maquette": ("Graphe",),
    }.items()
}


def duration_for(difficulty: object) -> float:
    """Return a difficulty's fixed duration and fail fast for unknown labels."""
    key = normal(difficulty)
    if key not in DURATION_BY_DIFFICULTY:
        raise KeyError(f"No duration configured for difficulty: {difficulty!r}")
    return DURATION_BY_DIFFICULTY[key]


def workflow_for(skill: object) -> tuple[str, ...]:
    """Return the ordered workflow for a project skill."""
    key = normal(skill)
    if key not in WORKFLOW_BY_SKILL:
        raise KeyError(f"No workflow configured for skill: {skill!r}")
    return WORKFLOW_BY_SKILL[key]

## 2. Load and validate data

The loader accepts comma- or semicolon-delimited data, only keeps valid
approved leave records, and reports issues before scheduling begins.

In [ ]:
def api_json(url: str) -> pd.DataFrame:
    """Fetch a Triweb API response and fail clearly when the live endpoint is unavailable."""
    response = requests.get(url, headers={"Accept": "application/json"}, timeout=45, verify=VERIFY_TLS)
    response.raise_for_status()
    payload = response.json()
    if not isinstance(payload, list):
        raise ValueError("Triweb API returned an unexpected JSON shape.")
    return pd.DataFrame(payload)


def api_skill(nature: object) -> str:
    """Map the API project nature to the existing business workflow skill."""
    label = normal(nature)
    if "correction" in label and "maj" in label:
        return "Corrections MAJ"
    if "correction" in label:
        return "Corrections CREA"
    if "sem" in label or "sea" in label:
        return "SEM/SEA"
    if "seo" in label or "referencement" in label or "audience" in label:
        return "SEO"
    if "logo" in label:
        return "Logo"
    if "maquette" in label:
        return "Maquette"
    if "webtool" in label:
        return "Webtool"
    if "toolbox" in label:
        return "Toolbox"
    if "ecommerce" in label:
        return "Ecommerce"
    if "oxatis" in label or "wizishop" in label:
        return "Oxatis"
    if "gmb" in label or "facebook" in label:
        return "GMB/Facebook"
    return "Creation Page"


def api_date(row: pd.Series) -> pd.Timestamp:
    """Create a project arrival date from the API day/month field and its year."""
    value, year = row.get("dateReception"), row.get("nYear")
    if pd.isna(value) or pd.isna(year):
        return pd.NaT
    return pd.to_datetime(f"{value}/{int(year)}", dayfirst=True, errors="coerce")


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load local employee scores, then build live projects and availability from Triweb APIs."""
    emp = pd.read_csv("employees.csv", sep=";").rename(columns={
        "id": "EmployeeID", "empname": "EmployeeName", "departement": "Department",
        "team_name": "TeamName", "subteam_name": "SubteamName",
    }).copy()
    required = {"EmployeeID", "EmployeeName", "Department", "Skills", "QualityScore", "RapidityScore", "Confidence"}
    if required - set(emp.columns):
        raise ValueError(f"employees.csv is missing columns: {sorted(required - set(emp.columns))}")
    emp["EmployeeID"] = emp["EmployeeID"].astype(str).str.strip()
    emp["SkillsSet"] = emp["Skills"].apply(skills)
    for column in ("QualityScore", "RapidityScore", "Confidence"):
        emp[column] = pd.to_numeric(emp[column], errors="coerce")
    employee_ids = set(pd.to_numeric(emp["EmployeeID"], errors="coerce").dropna().astype(int))

    availability = api_json(TRIWEB_AVAILABILITY_API)
    availability["userId"] = pd.to_numeric(availability["userId"], errors="coerce")
    availability = availability[availability["userId"].isin(employee_ids)].copy()
    availability["workDateIso"] = pd.to_datetime(availability["workDateIso"], errors="coerce")
    availability["AvailableHours"] = (pd.to_numeric(availability["morningMinutes"], errors="coerce").fillna(0) + pd.to_numeric(availability["afternoonMinutes"], errors="coerce").fillna(0)) / 60
    availability.to_csv(OUTPUT_DIR / "live_availability.csv", index=False)

    unavailable = availability[availability["AvailableHours"] < 7].copy()
    leaves = pd.DataFrame({
        "LeaveID": [f"API-{index}" for index in unavailable.index],
        "EmployeeID": unavailable["userId"].astype(int).astype(str),
        "EmployeeName": (unavailable["firstName"].fillna("") + " " + unavailable["lastName"].fillna("")).str.strip(),
        "LeaveType": unavailable["morningTypeInfo"].fillna(unavailable["afternoonTypeInfo"]).fillna("Unavailable"),
        "Start": unavailable["workDateIso"].dt.normalize() + pd.Timedelta(hours=8, minutes=30),
        "End": unavailable["workDateIso"].dt.normalize() + pd.Timedelta(hours=17),
        "Status": "Approved",
    }).dropna(subset=["Start"])

    raw = api_json(TRIWEB_PROJECTS_API)
    raw["CreationDate"] = raw.apply(api_date, axis=1)
    raw["Skill"] = raw["nature"].apply(api_skill)
    raw["DurationHours"] = (
        pd.to_numeric(raw["dureeR"], errors="coerce").fillna(0) + pd.to_numeric(raw["dureeG"], errors="coerce").fillna(0)
    ) / 3600
    completed = raw[raw["DurationHours"] > 0].copy()
    medians = completed.groupby("Skill")["DurationHours"].median().to_dict()
    fallback = {"Creation Page": 5, "Webtool": 8, "Toolbox": 8, "Ecommerce": 10, "GMB/Facebook": 2, "Oxatis": 8, "SEO": 1, "SEM/SEA": 1, "Logo": 3, "Maquette": 6, "Corrections CREA": 2, "Corrections MAJ": 1.5}

    global HISTORICAL_OWNERS
    HISTORICAL_OWNERS = {}
    for _, row in raw.iterrows():
        original = str(row.get("pq") if pd.notna(row.get("pq")) else row["id"])
        for stage, column in (("Redaction", "idRedacteur"), ("Graphe", "idGraphiste")):
            employee_id = pd.to_numeric(row.get(column), errors="coerce")
            if pd.notna(employee_id) and int(employee_id) in employee_ids:
                HISTORICAL_OWNERS[(original, stage)] = str(int(employee_id))

    today = pd.Timestamp.now().normalize()
    incoming = raw[(raw["CreationDate"] >= today) & (raw["CreationDate"] < today + pd.Timedelta(days=SIMULATION_DAYS))].copy()
    incoming = incoming[incoming[["idRedacteur", "idGraphiste"]].isna().all(axis=1)].copy()
    incoming["DurationHours"] = incoming["DurationHours"].where(incoming["DurationHours"] > 0, incoming["Skill"].map(medians))
    incoming["DurationHours"] = incoming["DurationHours"].fillna(incoming["Skill"].map(fallback)).fillna(5.0)
    projects = pd.DataFrame({
        "ProjectID": incoming["id"].astype(str), "OriginalProjectID": incoming["pq"].fillna(incoming["id"]).astype(str),
        "ClientCode": incoming["codeClient"].fillna("").astype(str), "CreationDate": incoming["CreationDate"], "Skill": incoming["Skill"],
        "Priority": pd.to_numeric(incoming["priorite"], errors="coerce").fillna(1).gt(1).map({True: "Urgent", False: "Normal"}),
        "Difficulty": incoming["nature"].fillna("API estimate"), "NombrePages": incoming["pages"].fillna(0),
        "CurrentStage": incoming["position"].fillna("Pending"), "Status": "Waiting", "DurationHours": incoming["DurationHours"],
    })
    manual_path = Path("manual_projects.csv")
    if manual_path.exists():
        manual = pd.read_csv(manual_path, sep=";")
        manual["CreationDate"] = pd.to_datetime(manual["CreationDate"], errors="coerce")
        manual["DurationHours"] = pd.to_numeric(manual.get("DurationHours", 5.0), errors="coerce").fillna(5.0)
        projects = pd.concat([projects, manual[projects.columns]], ignore_index=True)
    projects.to_csv(OUTPUT_DIR / "live_projects.csv", index=False)

    report = pd.DataFrame({"Issue": ["API availability rows retained", "Live unassigned projects", "Historical owner links"], "Rows": [len(availability), len(projects), len(HISTORICAL_OWNERS)]})
    return emp, projects, leaves, report


employees, projects, approved_leaves, quality_report = load_inputs()
EMPTY_LEAVES = approved_leaves.iloc[0:0]
LEAVES_BY_EMPLOYEE = {employee_id: group.sort_values("Start").reset_index(drop=True) for employee_id, group in approved_leaves.groupby("EmployeeID")}
QUALIFIED_EMPLOYEES: dict[tuple[str, str], list[pd.Series]] = defaultdict(list)
for _, employee_record in employees.iterrows():
    for skill_key in employee_record["SkillsSet"]:
        QUALIFIED_EMPLOYEES[(normal(employee_record["Department"]), skill_key)].append(employee_record)

display(Markdown("### Live API validation"))
display(quality_report)
print(f"Live availability retained for {availability['userId'].nunique():,} scored employees.")


## 3. Working calendar and weekly-capacity constraints

Working time is Monday–Friday, 08:30–12:00 and 13:30–17:00. The calendar
skips weekends and approved absences, and a task may never consume more than
the remaining capacity of its ISO week.

In [ ]:
def week(moment: pd.Timestamp) -> int:
    """Return the ISO week number stored in the WeeklyHours dictionary."""
    return int(moment.isocalendar().week)


def clock(day: pd.Timestamp, hour: int, minute: int) -> pd.Timestamp:
    """Make a timestamp at an exact time on a date."""
    return day.normalize() + pd.Timedelta(hours=hour, minutes=minute)


def work_periods(day: pd.Timestamp) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    """Return the two working intervals for a weekday and none at weekends."""
    if day.weekday() >= 5:
        return []
    return [(clock(day, start_h, start_m), clock(day, end_h, end_m)) for start_h, start_m, end_h, end_m in WORKING_PERIODS]


def active_leave(employee_id: str, instant: pd.Timestamp) -> pd.Series | None:
    """Return the approved leave covering an instant, if it exists."""
    employee_leaves = LEAVES_BY_EMPLOYEE.get(str(employee_id), EMPTY_LEAVES)
    matches = employee_leaves[(employee_leaves["Start"] <= instant) & (employee_leaves["End"] >= instant)]
    return None if matches.empty else matches.sort_values("End").iloc[0]


def next_work(employee_id: str, instant: pd.Timestamp) -> pd.Timestamp:
    """Move to the next working instant for an employee, avoiding approved leave."""
    cursor = pd.Timestamp(instant)
    while True:
        periods = work_periods(cursor)
        usable_period = next(((start, end) for start, end in periods if cursor < end), None)
        if usable_period is None:
            cursor = clock(cursor + pd.Timedelta(days=1), 8, 30)
            continue
        cursor = max(cursor, usable_period[0])
        leave = active_leave(employee_id, cursor)
        if leave is not None:
            cursor = pd.Timestamp(leave["End"]) + pd.Timedelta(minutes=1)
            continue
        return cursor


def next_monday(moment: pd.Timestamp) -> pd.Timestamp:
    """Return 08:30 on the Monday of the next ISO week."""
    return clock(moment + pd.Timedelta(days=7 - moment.weekday()), 8, 30)


def propose_slot(employee_id: str, earliest: pd.Timestamp, hours: float, weekly_hours: dict[int, float]) -> tuple[pd.Timestamp, pd.Timestamp, dict[int, float]]:
    """Propose a non-mutating time slot that observes all calendar constraints."""
    if hours <= 0:
        raise ValueError("Task duration must be positive.")
    cursor = next_work(employee_id, earliest)
    start, remaining = cursor, float(hours)
    used, allocation = defaultdict(float, weekly_hours), defaultdict(float)

    while remaining > 1e-9:
        cursor = next_work(employee_id, cursor)
        iso_week = week(cursor)
        available_week = MAX_WEEKLY_HOURS - used[iso_week]
        if available_week <= 1e-9:
            cursor = next_monday(cursor)
            continue

        period = next((item for item in work_periods(cursor) if item[0] <= cursor < item[1]), None)
        if period is None:
            cursor = next_work(employee_id, cursor)
            continue
        _, period_end = period
        employee_leaves = LEAVES_BY_EMPLOYEE.get(str(employee_id), EMPTY_LEAVES)
        upcoming = employee_leaves[(employee_leaves["Start"] > cursor) & (employee_leaves["Start"] < period_end)]
        usable_end = pd.Timestamp(upcoming["Start"].min()) if not upcoming.empty else period_end
        available_period = (usable_end - cursor).total_seconds() / 3600
        block = min(remaining, available_week, available_period)
        if block <= 1e-9:
            cursor = next_work(employee_id, usable_end + pd.Timedelta(minutes=1))
            continue
        cursor += pd.Timedelta(hours=block)
        remaining -= block
        used[iso_week] += block
        allocation[iso_week] += block
        if remaining > 1e-9:
            cursor = next_monday(cursor) if MAX_WEEKLY_HOURS - used[iso_week] <= 1e-9 else next_work(employee_id, cursor)

    return start, cursor, dict(allocation)

## 4. Eligibility and explainable scoring

Candidates must match the department and required skill, be feasible inside
the two-week planning horizon, and have a calendar slot. Scores remain in the
normalized 0–1 range until they are written to the output as percentages.

In [ ]:
def unit_score(value: object) -> float:
    """Normalize a supplied 0–1 or 0–100 score to the 0–1 interval."""
    score = float(pd.to_numeric(value, errors="coerce"))
    return 0.0 if not np.isfinite(score) else float(np.clip(score / 100 if score > 1 else score, 0, 1))


def performance(employee: pd.Series, priority: object) -> float:
    """Use the supplied historical scores with the required normal/urgent weights."""
    weights = (0.50, 0.30, 0.20) if normal(priority) == "urgent" else (0.80, 0.10, 0.10)
    return (
        weights[0] * unit_score(employee["QualityScore"])
        + weights[1] * unit_score(employee["RapidityScore"])
        + weights[2] * unit_score(employee["Confidence"])
    )


def candidates_for(project: pd.Series, stage: str, ready_at: pd.Timestamp, state: dict[str, dict], simulation_end: pd.Timestamp) -> list[dict]:
    """Filter qualified employees and calculate a feasible slot for each one."""
    records = []
    for employee in QUALIFIED_EMPLOYEES.get((normal(stage), project["SkillKey"]), []):
        employee_id = employee["EmployeeID"]
        values = state[employee_id]
        earliest = max(pd.Timestamp(project["CreationDate"]), pd.Timestamp(ready_at), values["AvailableAt"])
        start, end, allocation = propose_slot(employee_id, earliest, project["StageDuration"], values["WeeklyHours"])
        if start <= simulation_end:
            records.append({
                "Employee": employee, "EmployeeID": employee_id, "Start": start, "End": end,
                "WeeklyAllocation": allocation, "AssignedProjects": values["AssignedProjects"],
                "Capacity": max(0.0, 1 - values["WeeklyHours"].get(week(start), 0) / MAX_WEEKLY_HOURS),
            })
    return records


def rank(candidates: list[dict], priority: object) -> pd.DataFrame:
    """Apply performance, capacity, availability, and fairness scoring to candidates."""
    if not candidates:
        return pd.DataFrame()
    scores = pd.DataFrame(candidates)
    scores["Performance"] = scores["Employee"].apply(lambda item: performance(item, priority))
    delay = (scores["Start"] - scores["Start"].min()).dt.total_seconds()
    scores["Availability"] = 1.0 if delay.max() == 0 else 1 - delay / delay.max()
    maximum_assigned = scores["AssignedProjects"].max()
    scores["Fairness"] = 1.0 if maximum_assigned == 0 else 1 - scores["AssignedProjects"] / maximum_assigned
    scores["FinalScore"] = (
        0.40 * scores["Performance"] + 0.30 * scores["Capacity"]
        + 0.20 * scores["Availability"] + 0.10 * scores["Fairness"]
    )
    return scores.sort_values(["FinalScore", "Start", "EmployeeID"], ascending=[False, True, True]).reset_index(drop=True)

## 5. Chronological scheduler simulation

Correction work prefers an employee already recorded on the original project
and same department, provided that preference does not delay the work by more
than one working day. Other projects always select the highest final score.

In [ ]:
def initial_state(simulation_start: pd.Timestamp) -> dict[str, dict]:
    """Create the required dynamic scheduling state for all employees."""
    return {
        employee["EmployeeID"]: {
            "AvailableAt": next_work(employee["EmployeeID"], simulation_start),
            "CurrentProject": None, "WeeklyHours": {}, "AssignedHours": 0.0,
            "AssignedProjects": 0, "Status": "Available",
        }
        for _, employee in employees.iterrows()
    }


def select_correction(scored: pd.DataFrame, preferred_id: str | None) -> tuple[pd.Series, str]:
    """Prefer the original employee only when it keeps the delivery timely."""
    best = scored.iloc[0]
    if preferred_id is None:
        return best, "Best score"
    preferred = scored[scored["EmployeeID"] == str(preferred_id)]
    if preferred.empty:
        return best, "Original employee unavailable"
    candidate = preferred.iloc[0]
    if candidate["Start"] <= best["Start"] + pd.Timedelta(hours=8):
        return candidate, "Original employee preferred"
    return best, "Original employee would delay more than one working day"


def run_scheduler(project_source: pd.DataFrame, simulation_days: int = SIMULATION_DAYS) -> tuple[pd.DataFrame, pd.DataFrame, pd.Timestamp, pd.Timestamp]:
    """Simulate project arrivals in order and return planning and employee-state dataframes."""
    simulation_start = pd.Timestamp(project_source["CreationDate"].min())
    simulation_end = simulation_start.normalize() + pd.Timedelta(days=simulation_days - 1, hours=17)
    incoming = project_source[
        (project_source["CreationDate"] >= simulation_start) & (project_source["CreationDate"] <= simulation_end)
    ].sort_values(["CreationDate", "ProjectID"]).reset_index(drop=True)

    state, owners, records = initial_state(simulation_start), dict(HISTORICAL_OWNERS), []
    correction_keys = {normal("Corrections CREA"), normal("Corrections MAJ")}

    for _, raw_project in incoming.iterrows():
        project, ready_at = raw_project.copy(), pd.Timestamp(raw_project["CreationDate"])
        stages = workflow_for(project["Skill"])
        total = float(project.get("DurationHours", duration_for(project["Difficulty"])))
        correction = project["SkillKey"] in correction_keys

        for stage in stages:
            project["StageDuration"] = total if len(stages) == 1 else round(total * STAGE_EFFORT_SPLIT[stage], 2)
            scored = rank(candidates_for(project, stage, ready_at, state, simulation_end), project["Priority"])
            if scored.empty:
                records.append({
                    "ProjectID": project["ProjectID"], "OriginalProjectID": project["OriginalProjectID"], "ClientCode": project["ClientCode"],
                    "CreationDate": project["CreationDate"], "Stage": stage, "Department": stage, "EmployeeID": pd.NA, "EmployeeName": pd.NA,
                    "Skill": project["Skill"], "Priority": project["Priority"], "Difficulty": project["Difficulty"], "Start": pd.NaT, "End": pd.NaT,
                    "Duration": project["StageDuration"], "Status": "Waiting", "Performance": np.nan, "Capacity": np.nan,
                    "Availability": np.nan, "Fairness": np.nan, "FinalScore": np.nan, "CorrectionDecision": "No eligible employee before simulation end",
                    "WeeklyAllocation": {},
                })
                break

            preferred = owners.get((str(project["OriginalProjectID"]), stage)) if correction else None
            chosen, correction_note = select_correction(scored, preferred)
            employee_id, employee_state = chosen["EmployeeID"], state[chosen["EmployeeID"]]
            for iso_week, booked_hours in chosen["WeeklyAllocation"].items():
                employee_state["WeeklyHours"][int(iso_week)] = round(
                    employee_state["WeeklyHours"].get(int(iso_week), 0) + float(booked_hours), 2
                )
            employee_state["AssignedHours"] = round(employee_state["AssignedHours"] + float(project["StageDuration"]), 2)
            employee_state["AssignedProjects"] += 1
            employee_state["CurrentProject"] = project["ProjectID"]
            employee_state["AvailableAt"] = next_work(employee_id, chosen["End"] + pd.Timedelta(minutes=POST_TASK_BREAK_MINUTES))
            employee_state["Status"] = "Scheduled"
            owners[(str(project["ProjectID"]), stage)] = employee_id

            records.append({
                "ProjectID": project["ProjectID"], "OriginalProjectID": project["OriginalProjectID"], "ClientCode": project["ClientCode"],
                "CreationDate": project["CreationDate"], "Stage": stage, "Department": stage, "EmployeeID": employee_id,
                "EmployeeName": chosen["Employee"]["EmployeeName"], "Skill": project["Skill"], "Priority": project["Priority"],
                "Difficulty": project["Difficulty"], "Start": chosen["Start"], "End": chosen["End"], "Duration": float(project["StageDuration"]),
                "Status": "Assigned", "Performance": round(100 * chosen["Performance"], 2), "Capacity": round(100 * chosen["Capacity"], 2),
                "Availability": round(100 * chosen["Availability"], 2), "Fairness": round(100 * chosen["Fairness"], 2),
                "FinalScore": round(100 * chosen["FinalScore"], 2), "CorrectionDecision": correction_note,
                "WeeklyAllocation": dict(chosen["WeeklyAllocation"]),
            })
            ready_at = chosen["End"]

    planning = pd.DataFrame(records)
    state_rows = []
    for employee_id, values in state.items():
        employee = employees.loc[employees["EmployeeID"] == employee_id].iloc[0]
        state_rows.append({
            "EmployeeID": employee_id, "EmployeeName": employee["EmployeeName"], "Department": employee["Department"],
            "AvailableAt": values["AvailableAt"], "CurrentProject": values["CurrentProject"], "WeeklyHours": values["WeeklyHours"],
            "AssignedHours": values["AssignedHours"], "AssignedProjects": values["AssignedProjects"], "Status": values["Status"],
            "QualityScore": employee["QualityScore"], "RapidityScore": employee["RapidityScore"], "Confidence": employee["Confidence"],
        })
    employee_state = pd.DataFrame(state_rows).sort_values(["Department", "EmployeeName"]).reset_index(drop=True)
    return planning, employee_state, simulation_start, simulation_end


planning_df, employee_state_df, simulation_start, simulation_end = run_scheduler(projects)
planning_df.to_csv(OUTPUT_DIR / "planning.csv", index=False)
employee_state_df.assign(WeeklyHours=employee_state_df["WeeklyHours"].apply(json.dumps)).to_csv(
    OUTPUT_DIR / "employee_state.csv", index=False
)

required_columns = [
    "ProjectID", "OriginalProjectID", "ClientCode", "CreationDate", "Stage", "Department", "EmployeeID", "EmployeeName",
    "Skill", "Priority", "Difficulty", "Start", "End", "Duration", "Status", "Performance", "Capacity", "Availability",
    "Fairness", "FinalScore",
]
assert set(required_columns).issubset(planning_df.columns)

assigned = planning_df.query("Status == 'Assigned'").copy()
waiting = planning_df.query("Status == 'Waiting'").copy()
print(f"Simulation window: {simulation_start:%d %b %Y %H:%M} to {simulation_end:%d %b %Y %H:%M}")
print(f"Assigned stages: {len(assigned):,} | Waiting stages: {len(waiting):,}")
print(f"Exports: {OUTPUT_DIR / 'planning.csv'} and {OUTPUT_DIR / 'employee_state.csv'}")
display(planning_df[required_columns].sort_values(["Start", "EmployeeName"], na_position="last").head(25))
display(employee_state_df.head(25))

## 6. Dashboard and scheduling analytics

The Gantt supports native Plotly zoom, pan, range selection, hover, and
image export. The remaining charts provide workload, score, priority, and
two-week capacity analysis for the thesis demonstration.

In [ ]:
total_projects = planning_df["ProjectID"].nunique()
assigned_projects = assigned["ProjectID"].nunique()
utilization = employee_state_df["AssignedHours"].sum() / (len(employee_state_df) * 2 * MAX_WEEKLY_HOURS)
kpis = pd.DataFrame({
    "KPI": ["Incoming projects", "Assigned projects", "Waiting projects", "Employee utilization", "Average decision score"],
    "Value": [
        total_projects, assigned_projects, total_projects - assigned_projects,
        f"{utilization:.1%}", f"{assigned['FinalScore'].mean():.1f}/100",
    ],
})
display(Markdown("### Key performance indicators"))
display(kpis.style.hide(axis="index").set_properties(**{"font-size": "15px"}))

gantt = px.timeline(
    assigned.sort_values(["EmployeeName", "Start"]), x_start="Start", x_end="End", y="EmployeeName", color="Department",
    hover_data={"ProjectID": True, "ClientCode": True, "Skill": True, "Priority": True, "Duration": ":.2f", "FinalScore": ":.2f", "Stage": True},
    title="Production planning by employee", color_discrete_map={"Redaction": "#2563eb", "Graphe": "#f97316"},
)
gantt.update_yaxes(autorange="reversed", title=None)
gantt.update_xaxes(title=None, rangeslider_visible=True)
gantt.update_layout(height=max(520, 20 * assigned["EmployeeName"].nunique()), legend_title_text="Department")
gantt.show()

In [ ]:
department_workload = assigned.groupby("Department", as_index=False)["Duration"].sum().sort_values("Duration", ascending=False)
priority_counts = assigned.groupby("Priority", as_index=False).size()
employee_ranking = employee_state_df.sort_values("AssignedHours", ascending=False)

fig = px.bar(
    department_workload, x="Department", y="Duration", color="Department", text_auto=".1f",
    title="Department workload (hours)",
)
fig.update_layout(showlegend=False, yaxis_title="Scheduled hours")
fig.show()

fig = px.pie(priority_counts, names="Priority", values="size", hole=0.58, title="Assigned stages by priority")
fig.show()

fig = px.bar(
    employee_ranking.head(25), x="AssignedHours", y="EmployeeName", color="Department", orientation="h",
    hover_data=["AssignedProjects", "WeeklyHours"], title="Employee workload distribution (top 25)",
)
fig.update_yaxes(autorange="reversed", title=None)
fig.update_layout(xaxis_title="Assigned hours")
fig.show()

fig = px.histogram(assigned, x="FinalScore", nbins=20, color="Priority", barmode="overlay", title="Final decision-score distribution")
fig.update_layout(xaxis_title="Final score (0–100)")
fig.show()

score_components = assigned.melt(
    id_vars="ProjectID", value_vars=["Performance", "Capacity", "Availability", "Fairness"],
    var_name="Component", value_name="Score",
)
fig = px.box(score_components, x="Component", y="Score", color="Component", title="Score-component distribution")
fig.update_layout(showlegend=False, yaxis_title="Score (0–100)")
fig.show()

In [ ]:
weekly_records = []
for _, assignment in assigned.iterrows():
    for iso_week, booked_hours in assignment["WeeklyAllocation"].items():
        weekly_records.append({
            "EmployeeName": assignment["EmployeeName"], "Week": f"W{int(iso_week):02d}", "Hours": booked_hours,
        })
weekly_load = pd.DataFrame(weekly_records).groupby(["EmployeeName", "Week"], as_index=False)["Hours"].sum()
top_employees = employee_ranking.head(25)["EmployeeName"]
heatmap_data = weekly_load[weekly_load["EmployeeName"].isin(top_employees)].pivot(
    index="EmployeeName", columns="Week", values="Hours"
).fillna(0).reindex(top_employees).fillna(0)

heatmap = go.Figure(go.Heatmap(
    z=heatmap_data.to_numpy(), x=heatmap_data.columns, y=heatmap_data.index, colorscale="Blues", colorbar_title="Hours",
    hovertemplate="Employee=%{y}<br>Week=%{x}<br>Hours=%{z:.2f}<extra></extra>",
))
heatmap.update_layout(title="Weekly utilization heatmap (top 25 employees)", xaxis_title="ISO week", yaxis_title=None, height=650)
heatmap.show()

waiting_hours = (assigned["Start"] - assigned["CreationDate"]).dt.total_seconds() / 3600
statistics = pd.DataFrame({
    "Metric": ["Average waiting time", "Average stage duration", "Workload spread (standard deviation)", "Average capacity remaining at selection"],
    "Value": [
        f"{waiting_hours.mean():.2f} hours", f"{assigned['Duration'].mean():.2f} hours",
        f"{employee_state_df['AssignedHours'].std():.2f} hours", f"{assigned['Capacity'].mean():.2f}%",
    ],
})
display(Markdown("### Summary statistics"))
display(statistics.style.hide(axis="index"))

## 7. Re-run a scenario

Edit the constants or duration mapping, then run the notebook from top to
bottom. The deterministic engine will reproduce an auditable result for the
same inputs and policy. Planning exports are saved in the workspace outputs
folder.